# TI-Toolbox EEG and field workflow

Start with the fully synthetic checks below. The second part is a real-data workflow and deliberately stops unless `TIT_WORKFLOW_CONFIG` names an explicit external configuration. Select the TI-Toolbox/SimNIBS kernel for that part. This notebook ships without outputs or participant data.

# Synthetic EEG, field and graph examples

Run with a kernel containing the current TI-Toolbox, NumPy, SciPy and MNE. An installed toolbox is preferred; `TIT_SOURCE_DIR` may explicitly name a development checkout.

Everything here is generated in memory. No participant files, head models, downloads or output files are needed. The 256 permutations are a quick teaching example, not the settings used in the sleep study. The study modules remain responsible for cohort selection, event detection, contrasts, masks and inference families.

In [ ]:
import os
from pathlib import Path
import sys

# Use an installed TI-Toolbox, or explicitly select a development checkout.
if os.environ.get("TIT_SOURCE_DIR"):
    toolbox_source = Path(os.environ["TIT_SOURCE_DIR"]).expanduser().resolve()
    if not (toolbox_source / "tit").is_dir():
        raise FileNotFoundError("TIT_SOURCE_DIR must contain the tit package")
    sys.path.insert(0, str(toolbox_source))

import numpy as np
from scipy import sparse
from tit.fields import carrier_metrics
from tit.atlas.surface import parcel_means
from tit.stats.graph import cluster_permutation, correlation_cluster_permutation


## Carrier vectors and parcel summaries

At each of six vertices, both carrier vectors and the unit normal share one coordinate frame. Parallel, orthogonal and opposing pairs repeat twice. `hf_peak` is the worst-case carrier magnitude in V/m; `hf_sar` is a field-squared heating proxy, not calibrated SAR. `TI_normal` is the surface-normal envelope summary for two carriers.

Parcel labels align with the final vertex axis. These are two synthetic parcels, not an anatomical atlas.

In [ ]:
e1 = np.tile([1.0, 0.0, 0.0], (6, 1))
e2 = np.array([[0.5, 0, 0], [0, 1, 0], [-0.5, 0, 0]] * 2, dtype=float)
normals = np.tile([1.0, 0.0, 0.0], (6, 1))
fields = carrier_metrics([e1, e2], normals=normals)

# Independent geometric expectations, rather than copied implementation output.
np.testing.assert_allclose(fields["hf_peak"], [1.5, np.sqrt(2), 1.5] * 2)
np.testing.assert_allclose(fields["hf_sar"], [1.25, 2.0, 1.25] * 2)
np.testing.assert_allclose(fields["TI_normal"], [1.0, 0.0, 1.0] * 2)
labels = np.array([0, 0, 0, 1, 1, 1])
means, region_ids = parcel_means(fields["hf_peak"], labels)
np.testing.assert_array_equal(region_ids, [0, 1])
np.testing.assert_allclose(means, [(3 + np.sqrt(2)) / 3] * 2)
print("Carrier and parcel checks passed.")


## Participant-level graph inference

Rows represent independent synthetic participants; columns represent the six vertices. The line graph defines spatial neighbors. A paired-change test flips participant rows, and the field–response test permutes complete response rows. Each call controls its own graph family; it does not jointly correct the two demonstrations.

The first three vertices receive a strong simulated effect. Statistical maps describe this toy example only.

In [ ]:
rng = np.random.default_rng(314)
adjacency = sparse.diags([np.ones(5), np.ones(5)], [-1, 1], shape=(6, 6), format="csr")
changes = rng.normal(0, 0.25, size=(24, 6))
changes[:, :3] += 2.5
paired = cluster_permutation(
    changes, adjacency, statistic="t", n_permutations=256, seed=42,
    threshold_p=0.05, alpha=0.05, tail=0, sign_policy="separate",
)
assert paired.significant_mask[:3].all()
assert paired.null_distribution.shape == (256,)

exposure = rng.normal(size=(24, 6))
response = rng.normal(size=(24, 6))
response[:, :3] = exposure[:, :3] + rng.normal(0, 0.1, size=(24, 3))
association = correlation_cluster_permutation(
    exposure, response, adjacency, method="spearman",
    n_permutations=256, seed=43, threshold_p=0.05, alpha=0.05,
)
assert association.significant_mask[:3].all()
assert np.all((association.pvalues >= 0) & (association.pvalues <= 1))
print("Paired graph p-values:", paired.pvalues)
print("Correlation graph p-values:", association.pvalues)


## MNE events and explicit interval boundaries

This synthetic recording has a nonzero first sample. The generic annotation helper returns seconds relative to the first stored sample. Comparison windows are constructed explicitly; the helper does not remove artifacts, restore deleted time or clip intervals to the recording. These decisions belong to the analysis.

For real recordings, preserve the same MNE metadata and supply the actual markers and interval policy. Do not substitute these toy markers or durations into a study.

In [ ]:
import mne
from tit.eeg import annotation_pairs, stimulation_windows
from tit.source.reconstruction import epochs_from_intervals, diffusion_smoother

info = mne.create_info(["Fp1", "Fp2", "Cz", "Pz"], sfreq=100.0, ch_types="eeg")
raw = mne.io.RawArray(rng.normal(0, 1e-6, (4, 1000)), info, first_samp=500, verbose=False)
raw.set_annotations(mne.Annotations([2.0, 4.0], [0.0, 0.0], ["example start", "example end"]))
pairs = annotation_pairs(raw, "example start", "example end", relative_to_raw=True)
np.testing.assert_allclose(pairs, [(2.0, 4.0)])
windows = stimulation_windows(pairs, pre_duration=1.0, post_duration=1.0)
assert (windows[0].pre_start_sec, windows[0].post_end_sec) == (1.0, 5.0)
noise_epochs = epochs_from_intervals(raw, [(0.0, 2.0)], epoch_duration=1.0)
assert len(noise_epochs) == 2
np.testing.assert_array_equal(noise_epochs.events[:, 0], [500, 600])

# Neighbor averaging must preserve a constant surface map.
smoother = diffusion_smoother(adjacency, n_iters=2)
np.testing.assert_allclose(smoother @ np.ones(6), np.ones(6))
print("MNE timing, interval and smoothing checks passed.")


## Continue with actual data

Open `02_models_projection_source.ipynb` in the TI-Toolbox/SimNIBS kernel when you have explicit input files and approved configuration. The generic functions above do not select sleep stages, detect slow waves, define origin/involvement, or choose study contrasts. Keep those choices in the study modules; avoid copying their scientific settings into notebook cells.

# Head model → simulation → projection → EEG source windows

This notebook calls TI-Toolbox directly inside its SimNIBS environment. It can run structural preprocessing, configured FEM simulations, a recording-aligned EEG forward model, carrier projection and explicit inverse windows. It does not invent a montage, current, coordinate transform, reference, frequency band or event selection.

Set `TIT_WORKFLOW_CONFIG` to an external JSON file. Paths in that file are absolute or relative to the JSON's directory. Keep subject metadata, recordings and derived outputs outside the source repository. Running these cells performs the selected work; preprocessing and FEM can take substantial time.

The required JSON keys are:

- `project_root`, `subject_id` (without `sub-`), `head_model_dir`.
- `preprocessing`: explicit `run_pipeline` keyword arguments, or `null` to reuse the supplied existing head model.
- `simulation_config_json`: a saved serialized `SimulationConfig`; `simulation_overwrite`: explicit boolean.
- `forward_config_json`: a saved serialized `ForwardConfig`; `recording_fif`: prepared MNE recording; `trans_fif`: supplied head-to-MRI transform or `null` to use the recording's digitized fiducials; `forward_output_dir`.
- `carrier_meshes`: explicit list of carrier E-vector mesh paths; `fsaverage_spacing`; `projected_fields_npz`.
- `source`: `noise_intervals_seconds`, `exclusions_seconds`, `epoch_duration_seconds`, `covariance_method`, `rank`, `loose`, `depth`, `peak_samples_npy`, `offset_samples_npy`, `lambda2`, `method`, `pick_ori`, `output_npz`.

Use serialized configs from your actual project. The notebook requires all simulation/forward config fields to be present, so omitted scientific settings cannot silently acquire library defaults. The recording must already carry the intended filtering, reference, bad-channel annotations and digitization.

In [ ]:
import json
import os
from pathlib import Path

manifest_name = os.environ.get("TIT_WORKFLOW_CONFIG")
if not manifest_name:
    raise RuntimeError("Set TIT_WORKFLOW_CONFIG to your external workflow JSON before running real-data cells.")
manifest = Path(manifest_name).expanduser().resolve()
if not manifest.is_file():
    raise FileNotFoundError(f"Workflow JSON not found: {manifest}")
job = json.loads(manifest.read_text())
required = {"project_root", "subject_id", "head_model_dir", "preprocessing",
            "simulation_config_json", "simulation_overwrite", "forward_config_json",
            "recording_fif", "trans_fif", "forward_output_dir", "carrier_meshes",
            "fsaverage_spacing", "projected_fields_npz", "source"}
if missing := required - job.keys():
    raise ValueError(f"Workflow JSON is missing keys: {sorted(missing)}")

def resolve_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else manifest.parent / path).resolve()

project_root = resolve_path(job["project_root"])
if not project_root.is_dir():
    raise FileNotFoundError(f"Project root not found: {project_root}")
subject_id = str(job["subject_id"])
if not subject_id or subject_id.startswith("sub-"):
    raise ValueError("subject_id must be the explicit bare subject label (without sub-)")
head_model = resolve_path(job["head_model_dir"])


In [ ]:
import os
from pathlib import Path
import sys

# Use an installed TI-Toolbox, or explicitly select a development checkout.
if os.environ.get("TIT_SOURCE_DIR"):
    toolbox_source = Path(os.environ["TIT_SOURCE_DIR"]).expanduser().resolve()
    if not (toolbox_source / "tit").is_dir():
        raise FileNotFoundError("TIT_SOURCE_DIR must contain the tit package")
    sys.path.insert(0, str(toolbox_source))

from dataclasses import fields as dataclass_fields
import inspect
import numpy as np
import mne
from tit.paths import reset_path_manager, get_path_manager
from tit.config_io import deserialize_config
from tit.pre import run_pipeline
from tit.sim import SimulationConfig, run_simulation
from tit.source import ForwardConfig, prepare_forward, project_carrier_fields
from tit.source.reconstruction import covariance_from_intervals, prepare_inverse, apply_inverse_windows

# Select this explicitly supplied project for the current notebook kernel.
reset_path_manager()
pm = get_path_manager(str(project_root))
if Path(pm.m2m(subject_id)).resolve() != head_model:
    raise ValueError("run_simulation uses the selected project's m2m directory; "
                     "head_model_dir must refer to that same model for this workflow")

def read_config(cls, filename):
    contents = json.loads(resolve_path(filename).read_text())
    expected = {field.name for field in dataclass_fields(cls) if field.init}
    if missing := expected - contents.keys():
        raise ValueError(f"Supply a complete serialized {cls.__name__}: missing {sorted(missing)}")
    return deserialize_config(cls, contents, strict=True)

simulation_config = read_config(SimulationConfig, job["simulation_config_json"])
forward_config = read_config(ForwardConfig, job["forward_config_json"])
if simulation_config.subject_id.removeprefix("sub-") != subject_id:
    raise ValueError("Workflow and simulation config identify different subjects")
if type(job["simulation_overwrite"]) is not bool:
    raise ValueError("simulation_overwrite must be an explicit JSON boolean")


## Build or reuse the supplied head model

A `null` preprocessing configuration explicitly requests reuse and verifies the directory exists. Otherwise, at least one preprocessing step must be selected. `run_pipeline` uses the chosen TI-Toolbox project layout; the supplied `head_model_dir` must point to its intended output.

In [ ]:
preprocessing = job["preprocessing"]
if preprocessing is None:
    if not head_model.is_dir():
        raise FileNotFoundError(f"Requested existing head model is absent: {head_model}")
    print("Using the supplied existing head model; no preprocessing was requested.")
else:
    step_names = ("convert_dicom", "create_m2m", "run_fastsurfer", "run_freesurfer",
                  "run_tissue_analysis", "run_qsiprep", "run_qsirecon", "extract_dti")
    if not isinstance(preprocessing, dict) or not any(preprocessing.get(k) is True for k in step_names):
        raise ValueError("Select at least one preprocessing step, or set preprocessing to null.")
    inspect.signature(run_pipeline).bind([subject_id], **preprocessing)
    status = run_pipeline([subject_id], **preprocessing)
    if status != 0:
        raise RuntimeError(f"Structural pipeline failed with status {status}")
    if not head_model.is_dir():
        raise FileNotFoundError(f"Configured head-model output was not produced: {head_model}")


## Simulate the saved montage and current configuration

The serialized configuration is the authority for electrode geometry, conductivities and delivered currents. No stimulation settings are supplied by this notebook. Inspect the returned simulation results before selecting the explicit carrier meshes used below.

In [ ]:
simulation_results = run_simulation(simulation_config, overwrite=job["simulation_overwrite"])
if not simulation_results:
    raise RuntimeError("Simulation returned no results")
simulation_results


## Prepare the forward solution with recording metadata

The recorded `Info` supplies channel identities and digitization. A supplied head-to-MRI transform takes precedence; otherwise TI-Toolbox uses the recording's fiducials. This cell does not estimate a guessed transform or substitute a generic EEG cap for missing recording metadata.

In [ ]:
recording_path = resolve_path(job["recording_fif"])
if not recording_path.is_file():
    raise FileNotFoundError(f"Prepared recording not found: {recording_path}")
raw = mne.io.read_raw_fif(recording_path, preload=True, verbose=False)
trans = None if job["trans_fif"] is None else mne.read_trans(resolve_path(job["trans_fif"]))
forward_path, source_path, morph_path = prepare_forward(
    subject_id, forward_config,
    output_dir=resolve_path(job["forward_output_dir"]), head_model_dir=head_model,
    info=raw.info, trans=trans,
)
assert all(Path(path).is_file() for path in (forward_path, source_path, morph_path))


## Project the explicit carrier vector fields

Carrier vectors are interpolated onto a common native central surface, combined there, then scalar summaries are morphed to fsaverage. The output concatenates left then right hemisphere vertices. This preserves the order of vector operations; it is not equivalent to combining independently morphed scalar magnitudes. No projected file is silently overwritten.

In [ ]:
carrier_meshes = [resolve_path(path) for path in job["carrier_meshes"]]
if len(carrier_meshes) < 2 or not all(path.is_file() for path in carrier_meshes):
    raise FileNotFoundError("Supply at least two existing carrier E-vector meshes")
projected_path = resolve_path(job["projected_fields_npz"])
if projected_path.exists():
    raise FileExistsError(f"Choose a new output path or explicitly remove the prior output: {projected_path}")
projected = project_carrier_fields(carrier_meshes, head_model, spacing=job["fsaverage_spacing"])
projected_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(projected_path, **projected)
print("Projected fields:", {name: values.shape for name, values in projected.items()})


## Reconstruct explicitly selected EEG windows

Noise intervals and exclusions are seconds relative to the stored recording. Peak indices and offsets are integer samples on that same recording, after all intended resampling. They are supplied externally; this notebook does not detect or merge events. The source result retains vertex × event × time dimensions (plus orientation when requested). Choose inverse parameters in the external configuration.

Origin/involvement rules, sleep-stage selection, historical timing conventions, outcome definitions and statistical families remain in study modules. Do not treat this generic reconstruction as a replacement for those adapters.

In [ ]:
source = job["source"]
noise_cov = covariance_from_intervals(
    raw, source["noise_intervals_seconds"], exclusions=source["exclusions_seconds"],
    epoch_duration=source["epoch_duration_seconds"], method=source["covariance_method"],
    rank=source["rank"], legacy_sampling=False, reject_by_annotation=True,
)
forward = mne.read_forward_solution(forward_path, verbose=False)
inverse = prepare_inverse(raw.info, forward, noise_cov, loose=source["loose"],
                          depth=source["depth"], rank=source["rank"])
peak_samples = np.load(resolve_path(source["peak_samples_npy"]), allow_pickle=False)
offsets = np.load(resolve_path(source["offset_samples_npy"]), allow_pickle=False)
morph = mne.read_source_morph(morph_path)
source_windows = apply_inverse_windows(
    raw.get_data(), raw.info, inverse, peak_samples, offsets,
    lambda2=source["lambda2"], method=source["method"], pick_ori=source["pick_ori"], morph=morph,
)
source_output = resolve_path(source["output_npz"])
if source_output.exists():
    raise FileExistsError(f"Choose a new source output path: {source_output}")
source_output.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(source_output, source_windows=source_windows,
                    peak_samples=peak_samples, offset_samples=offsets,
                    sfreq=raw.info["sfreq"])
print("Source-window shape:", source_windows.shape)
